In [ ]:
import s3fs
import xarray as xr
import zarr

import numpy as np

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

## Local

In [ ]:
ds = xr.open_zarr("earthcare_ACM_CAP_2B_01164E.zarr")
ds

## S3

In [1]:
import s3fs
import xarray as xr
import zarr

store = s3fs.S3Map(
    root="obs/earthcare/ACM_CAP_2B_2024-08-11/level_11.zarr",
    s3=s3fs.S3FileSystem(endpoint_url="https://s3.waterpark.dkrz.de", anon=True)
)
ds = xr.open_zarr(store)
ds

<xarray.Dataset> Size: 403MB
Dimensions:         (cell: 50331648)
Coordinates:
    crs             float64 8B ...
Dimensions without coordinates: cell
Data variables:
    ice_water_path  (cell) float64 403MB dask.array<chunksize=(196608,), meta=np.ndarray>
Attributes:
    healpix_level:                11
    healpix_nside:                2048
    healpix_order:                nested
    grid_doctor_version:          2604.0.0
    grid_doctor_method:           binned-mean
    grid_doctor_implicit_coords:  1

## Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6),
                         subplot_kw={"projection": ccrs.PlateCarree()},
                         gridspec_kw={"width_ratios": [1, 1]})

# Ground track colored by HEALPix cell
ax = axes[0]
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3)
ax.gridlines(draw_labels=True, linewidth=0.3)
sc = ax.scatter(
    ds["longitude"].values, ds["latitude"].values,
    c=ds.cell_ids, s=1, cmap="tab20",
    transform=ccrs.PlateCarree(),
)
ax.set_title(f"Ground track — HEALPix depth {ds.healpix_level}")
ax.set_extent([
    float(ds.longitude.min()) - 2, float(ds.longitude.max()) + 2,
    float(ds.latitude.min()) - 2, float(ds.latitude.max()) + 2,
], crs=ccrs.PlateCarree())

# Curtain plot — find first 2D variable
ax = axes[1]
plot_var = None
for v in ds.data_vars:
    if ds[v].dims == ("along_track", "vertical") and v not in ("height", "range"):
        plot_var = v
        break

if plot_var and "height" in ds:
    data = ds[plot_var].values
    height = ds["height"].values / 1000  # km
    step = max(1, data.shape[0] // 500)
    im = ax.pcolormesh(
        np.arange(0, data.shape[0], step),
        height[::step, :].T,
        data[::step, :].T,
        shading="nearest", cmap="viridis",
    )
    ax.set_xlabel("Along-track index")
    ax.set_ylabel("Height (km)")
    ax.set_title(f"{plot_var} (curtain)")
    plt.colorbar(im, ax=ax, shrink=0.7)

plt.tight_layout()
plt.show()